# Lab 2: Model Optimization Techniques - Part C: Knowledge Distillation

Welcome to lab 2 of the TinyML course! In this lab, we are going to learn a few approaches for deep learning model optimization. These techniques, when combined, can significantly shrink model size and make them suitable for model deployment. The techniques we are going to cover today are:

* Knowledge Distillation
* Model Pruning
* Quantization

Please follow Part A (Quantization) to set up the environment and install dependencies. At the beginning, we again apply the workaround from part A to resolve compatibility issues between `tensorflow`, `keras` and `tensorflow-model-optimization`:

In [ ]:
import os

# Apply workaround before imports
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_USE_LEGACY_KERAS"] = "1"

## Knowledge Distillation / Pruning / Quantization

This notebook demonstrates how to combine **model compression techniques**—Knowledge Distillation (KD), Pruning, and Quantization—to create compact and efficient deep learning models suitable for deployment on resource-constrained devices.

### 🧠 Knowledge Distillation (KD)
Knowledge Distillation is a technique where a smaller model (student) is trained to mimic a larger, pre-trained model (teacher). The student learns from the teacher's output probabilities (soft labels), providing richer supervision than traditional hard labels.

### ✂️ Pruning
Pruning removes less important connections (weights) from the student model, introducing sparsity to reduce model size and inference cost.

### 🧮 Quantization
Quantization reduces the numerical precision of weights and activations. In this notebook, we apply **Float16** or **Integer Quantization** to the pruned student model to further compress it.

### What This Notebook Demonstrates:
1. Training a **teacher model** on the MNIST dataset.
2. Training a smaller **student model** using **knowledge distillation**.
3. Applying **pruning** to the student model to introduce sparsity.
4. Applying **quantization** (e.g., Float16 or INT8) to the pruned student model.
5. Comparing model sizes and accuracies of:
   - Teacher Model  
   - Student Model    
   - Quantized + Pruned Student Model


In [ ]:
# Import required libraries
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize the data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape the data for CNN input
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(f"Training data shape: {x_train.shape}, Labels shape: {y_train.shape}")
print(f"Test data shape: {x_test.shape}, Labels shape: {y_test.shape}")


In [ ]:
from tensorflow.data import Dataset
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Build a simple CNN model as the teacher
def create_teacher_model():
    return Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

# Create training and validation datasets
train_dataset = Dataset.from_tensor_slices((x_train, y_train))
val_dataset = Dataset.from_tensor_slices((x_test, y_test))
batch_size = 32

# Compile and train the model
teacher_model = create_teacher_model()
teacher_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
teacher_history = teacher_model.fit(train_dataset.batch(batch_size), epochs=2, validation_data=val_dataset.batch(batch_size))

# Get validation accuracy
teacher_accuracy = teacher_history.history["val_accuracy"][-1]


## Knowledge Distillation

The student model is trained using Knowledge Distillation. Instead of directly using the ground-truth labels, the student learns from the teacher's output probabilities (soft labels) using the Kullback-Leibler (KL) divergence.

In [ ]:
# Define the student model
def create_student_model():
    return Sequential([
        # Note we are using fewer channels here
        Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        MaxPooling2D((2, 2)),
        Flatten(),
        # Note we are using a smaller linear layer
        Dense(64, activation='relu'),
        Dense(10, activation='softmax')
    ])

student_model = create_student_model()

# Precompute teacher predictions (soft labels)
temperature = 5
teacher_soft_labels = teacher_model.predict(x_train, batch_size=32) / temperature

# Combine ground-truth labels and teacher soft labels
teacher_labels_dataset = Dataset.from_tensor_slices(teacher_soft_labels)
kd_dataset = Dataset.zip((train_dataset, teacher_labels_dataset))

In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.losses import CategoricalCrossentropy, KLDivergence
from tensorflow.keras.metrics import Mean
from tensorflow import GradientTape, nn

# Custom Knowledge Distillation training loss (used only during train_step)
def kd_loss(y_true, y_pred, soft_labels, temperature):
    # Hard label loss (ground truth)
    hard_loss = CategoricalCrossentropy()(y_true, y_pred)

    # Soft label loss (teacher predictions for the batch)
    soft_loss = KLDivergence()(
        nn.softmax(soft_labels / temperature),
        nn.softmax(y_pred / temperature)
    )

    # Standard KD formulation scales soft loss by temperature^2
    combined_loss = 0.5 * hard_loss + 0.5 * (temperature ** 2) * soft_loss

    return hard_loss, soft_loss, combined_loss

class KnowledgeDistiller(Model):
    def __init__(self, student, temperature):
        super().__init__()

        self.student = student
        self.temperature = temperature

        # Track each KD loss component per epoch
        self.hard_loss_tracker = Mean(name="hard_loss")
        self.soft_loss_tracker = Mean(name="soft_loss")
        self.kd_loss_tracker = Mean(name="kd_loss")

    @property
    def metrics(self):
        # Include compiled metrics + custom KD loss trackers
        return super().metrics + [
            self.hard_loss_tracker,
            self.soft_loss_tracker,
            self.kd_loss_tracker,
        ]

    def call(self, inputs, training=False):
        return self.student(inputs, training=training)

    def train_step(self, data):
        (x_batch, y_batch), soft_labels = data

        # Forward pass: compute KD losses
        with GradientTape() as tape:
            y_pred = self(x_batch, training=True)
            hard_loss, soft_loss, combined_loss = kd_loss(
                y_batch, y_pred, soft_labels, temperature=self.temperature
            )

        # Backward pass: compute gradients and weight updates
        grads = tape.gradient(combined_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        # Update custom loss trackers
        self.hard_loss_tracker.update_state(hard_loss)
        self.soft_loss_tracker.update_state(soft_loss)
        self.kd_loss_tracker.update_state(combined_loss)

        # Update compiled metrics (e.g., accuracy) using non-deprecated path
        metric_results = self.compute_metrics(
            x_batch, y_batch, y_pred, sample_weight=None
        )

        return {
            **metric_results,
            "loss": self.kd_loss_tracker.result(),
            "hard_loss": self.hard_loss_tracker.result(),
            "soft_loss": self.soft_loss_tracker.result()
        }

# Compile and train the student model in the wrapper
# (Note: loss only applies for evaluation because we overrode it in `train_step`)
distiller = KnowledgeDistiller(student_model, temperature=temperature)
distiller.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
distiller.fit(kd_dataset.batch(batch_size), epochs=2, verbose=1)

In [ ]:
# Evaluate the student model
student_accuracy = distiller.evaluate(val_dataset.batch(batch_size), verbose=0)[1]
print(f"Student Model Accuracy: {student_accuracy:.4f}")

---

### ✂️➕🔢 Pruning + Quantization Task

Now that you have successfully trained the **student model using Knowledge Distillation**, your next task is to **compress** it further for deployment.

Using your knowledge from the previous tasks related to quantization and pruning:

1. **Apply pruning** to the student model to introduce sparsity.
2. **Convert** the pruned model to TensorFlow Lite format using **full integer (INT8) quantization**.
   - Be sure to include a **representative dataset** for calibration.
   - Use appropriate settings to ensure the model is both **sparse** and **fully quantized**.

Finally:

- **Save your model** as `pruned_quantized_student_model.tflite`.
- This version should be **smaller** and suitable for deployment on **resource-constrained hardware**.

---


---

### 🧪 Lab 2 Submission Reminder

**Please complete the code below and take a screenshot of it as part of your Lab 2 submission.**

⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️

---

In [ ]:
# [ TODO ]
# 1. Apply pruning to the trained student model
# <--- Your code here to prune the student model --->
raise NotImplementedError

# 2. Convert the pruned model to TFLite using full integer quantization
# (Don't forget to provide a representative dataset and use appropriate settings for optimizations)
# <--- Your code here to set up the TFLite converter with INT8 quantization --->
raise NotImplementedError


---
⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️

### 📸 Lab 2 Submission Reminder

**Please take a screenshot of the above result and include it as part of your Lab 2 submission.**

---


## Summary

- The student model achieves comparable accuracy to the teacher model while being significantly smaller.
- Full integer quantization reduces the student model size further, with a slight accuracy drop.
- Knowledge Distillation enables smaller models to effectively mimic larger, more complex models.